In [ ]:
!pip install alphagenome

In [ ]:
from alphagenome import colab_utils
from alphagenome.data import gene_annotation, genome, track_data, transcript
from alphagenome.models import dna_client
from alphagenome.visualization import plot_components
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
print("Initializing AlphaGenome model...")
try:
    model = dna_client.create(colab_utils.get_api_key())
    print("[SUCCESS] Connected to AlphaGenome Cloud Backend API.")
except Exception as e:
    print(f"Fallback: {e}")
    from alphagenome.models import model_utils
    model = model_utils.load_model_local()

ONE_MB = 1048576

In [ ]:
output_metadata = model.output_metadata(
    dna_client.Organism.HOMO_SAPIENS
).concatenate()

In [ ]:
output_metadata.columns

In [ ]:
human_tracks = (
    model.output_metadata(dna_client.Organism.HOMO_SAPIENS)
    .concatenate()
    .groupby('output_type')
    .size()
    .rename('# Human tracks')
)

human_tracks

In [ ]:
# see all metadata with ontology info
meta = model.output_metadata(dna_client.Organism.HOMO_SAPIENS).concatenate()
print(meta.columns.tolist())  # see what columns exist first

In [ ]:
import inspect
print(inspect.signature(model.predict_variant))

In [ ]:
meta = model.output_metadata(dna_client.Organism.HOMO_SAPIENS).concatenate()

# PBMC/iPSC reprogramming relevant terms
search_terms = [
    'iPSC', 'ipsc', 'induced pluripotent',
    'PBMC', 'pbmc', 'peripheral blood mononuclear',
    'stem cell', 'pluripotent',
    'fibroblast',          # common reprogramming intermediate
    'lymphoblast',         # PBMC-derived
    'T cell', 'T-cell',   # major PBMC component
    'B cell', 'B-cell',   # major PBMC component
    'NK cell', 'natural killer',
    'monocyte',            # major PBMC component
    'lymphocyte',
    'WTC', 'GM12878', 'GM23338',  # known iPSC lines in ENCODE
    'K562',                # hematopoietic
    'Jurkat',
    'hematopoietic',
]

mask = meta[['biosample_name', 'biosample_type']].apply(
    lambda col: col.astype(str).str.contains(
        '|'.join(search_terms), case=False, na=False
    )
).any(axis=1)

hits = meta[mask][['ontology_curie', 'biosample_name',
                   'biosample_type', 'output_type']].drop_duplicates()

print(hits.to_string())

In [ ]:
print("\n=== MODALITY COVERAGE PER CELL TYPE ===")
coverage = (
    hits.groupby(['ontology_curie', 'biosample_name'])['output_type']
    .apply(lambda x: set(x.astype(str)))
    .reset_index()
)

modalities_of_interest = {
    'OutputType.RNA_SEQ',
    'OutputType.ATAC',
    'OutputType.CHIP_TF',
    'OutputType.CHIP_HISTONE',
    'OutputType.DNASE',
}

coverage['available_modalities'] = coverage['output_type'].apply(
    lambda x: x & modalities_of_interest
)
coverage['n_modalities'] = coverage['available_modalities'].apply(len)
coverage = coverage.sort_values('n_modalities', ascending=False)
print(coverage[['ontology_curie', 'biosample_name',
                'available_modalities', 'n_modalities']].to_string())

In [ ]:
MODALITY_CONFIG = [
    {
        "modality": dna_client.OutputType.RNA_SEQ,
        "attr":     "rna_seq",
        "label":    "RNA_SEQ",
        "ontology_terms": [
            'EFO:0009747',  # WTC11 — iPSC line derived from PBMCs (most direct proxy)
            'EFO:0007950',  # GM23338 — iPSC line
            'CL:2000001',   # peripheral blood mononuclear cell (direct PBMC)
            'CL:0000624',   # CD4+ T cell (major PBMC component)
            'CL:0000625',   # CD8+ T cell (major PBMC component)
            'CL:0000236',   # B cell (major PBMC component)
            'EFO:0002784',  # GM12878 — lymphoblastoid from B cells
        ],
    },
    {
        "modality": dna_client.OutputType.ATAC,
        "attr":     "atac",
        "label":    "ATAC",
        "ontology_terms": [
            'EFO:0009747',  # WTC11 — iPSC, most relevant
            'EFO:0007950',  # GM23338 — iPSC
            'EFO:0002784',  # GM12878 — lymphoblastoid
            'CL:0000624',   # CD4+ T cell
            'CL:0000625',   # CD8+ T cell
            'CL:0000236',   # B cell
        ],
    },
    {
        "modality": dna_client.OutputType.CHIP_TF,
        "attr":     "chip_tf",
        "label":    "CHIP_TF",
        "ontology_terms": [
            'EFO:0009747',  # WTC11 — iPSC
            'EFO:0007950',  # GM23338 — iPSC
            'EFO:0002784',  # GM12878 — lymphoblastoid
            'CL:0000236',   # B cell
            'CL:0000624',   # CD4+ T cell
            'CL:0000625',   # CD8+ T cell
        ],
    },
]

In [ ]:
# Test on one variant
test_interval = genome.Interval(chromosome='chr1', start=225000000, end=226048576)
test_variant  = genome.Variant(chromosome='chr1', position=225519331,
                                reference_bases='A', alternate_bases='ATCCAGGCGTTCCTGCCGC')

preds = model.predict_variant(
    interval=test_interval,
    variant=test_variant,
    organism=dna_client.Organism.HOMO_SAPIENS,
    requested_outputs=[dna_client.OutputType.RNA_SEQ,
                       dna_client.OutputType.ATAC,
                       dna_client.OutputType.CHIP_TF],
    ontology_terms=['EFO:0009747', 'EFO:0007950', 'EFO:0002784',
                    'CL:0000624', 'CL:0000625', 'CL:0000236', 'CL:2000001'],
)

print("RNA-seq shape:", preds.reference.rna_seq.values.shape)
print("ATAC shape:",    preds.reference.atac.values.shape)
print("CHIP-TF shape:", preds.reference.chip_tf.values.shape)

In [ ]:
# =====================================================================
# ALPHAGENOME BACKGROUND VARIANT VALIDATION PIPELINE
# Checkpointed, resumable, with statistical comparison
# =====================================================================
import numpy as np
import pandas as pd
import os
import json
import time
from scipy import stats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from google.colab import files, drive
from alphagenome import colab_utils
from alphagenome.data import genome
from alphagenome.models import dna_client

# ── Mount Drive for persistent checkpointing ─────────────────────────
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/alphagenome_background_all"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'checkpoint_all.json')
RESULTS_FILE    = os.path.join(CHECKPOINT_DIR, 'results_background_all.csv')
print(f"Checkpoint directory: {CHECKPOINT_DIR}")

# ── Initialize model ─────────────────────────────────────────────────
print("Initializing AlphaGenome model...")
try:
    model = dna_client.create(colab_utils.get_api_key())
    print("[SUCCESS] Connected to AlphaGenome Cloud Backend API.")
except Exception as e:
    print(f"Fallback: {e}")
    from alphagenome.models import model_utils
    model = model_utils.load_model_local()

# ── Constants ────────────────────────────────────────────────────────
ONE_MB = 1048576
CHECKPOINT_EVERY = 10

# Regulatory cCRE types to keep (exclude purely coding variants)
REGULATORY_CCRE = ['PLS', 'pELS', 'dELS', 'CTCF-bound',
                   'PLS,CTCF-bound', 'pELS,CTCF-bound', 'dELS,CTCF-bound']

# Per-modality proxy config (same as your 27-variant run)
MODALITY_CONFIG = [
    {
        "modality": dna_client.OutputType.RNA_SEQ,
        "attr":     "rna_seq",
        "label":    "RNA_SEQ",
        "ontology_terms": [
            'CL:0001054',  # CD14-positive monocyte
            'CL:0000842',  # mononuclear cell
            'CL:2000001',  # peripheral blood mononuclear cell
            'CL:0000837',  # hematopoietic multipotent progenitor
            'CL:0001059',  # common myeloid progenitor
        ],
    },
    {
        "modality": dna_client.OutputType.ATAC,
        "attr":     "atac",
        "label":    "ATAC",
        "ontology_terms": [
            'UBERON:0002107',  # liver
            'EFO:0002067',     # K562
        ],
    },
    {
        "modality": dna_client.OutputType.CHIP_TF,
        "attr":     "chip_tf",
        "label":    "CHIP_TF",
        "ontology_terms": [
            'CL:0001054',  # CD14-positive monocyte
            'EFO:0002793', # HL-60
        ],
    },
]

# ── Load and prepare CSVs ────────────────────────────────────────────
print("\nUpload your HIGH background CSV first, then LOW background CSV...")
uploaded = files.upload()
filenames = list(uploaded.keys())

if len(filenames) < 2:
    raise ValueError("Please upload both high and low CSV files.")

# Assign group based on filename
dfs = []
for fname in filenames:
    df = pd.read_csv(fname)
    if 'high' in fname.lower():
        df['group'] = 'High-Background'
    elif 'low' in fname.lower():
        df['group'] = 'Low-Background'
    else:
        # fallback: ask
        group = input(f"Is '{fname}' high or low infectivity? (high/low): ").strip().lower()
        df['group'] = 'High-Background' if group == 'high' else 'Low-Background'
    dfs.append(df)
    print(f"  Loaded {fname}: {len(df)} variants → {df['group'].iloc[0]}")

df_all = pd.concat(dfs, ignore_index=True)
print(f"\nTotal variants loaded: {len(df_all)}")

# ── Filter to regulatory variants only ───────────────────────────────
df_reg = df_all[df_all['cCRE_Type'].isin(REGULATORY_CCRE)].copy()
print(f"After regulatory filter: {len(df_reg)} variants")

# ── Derive variant type and unique ID ────────────────────────────────
def variant_type(ref, alt):
    if len(alt) > len(ref):   return 'Insertion'
    if len(ref) > len(alt):   return 'Deletion'
    return 'SNV'

df_reg['type']       = df_reg.apply(lambda r: variant_type(str(r['REF']), str(r['ALT'])), axis=1)

# ── Handle multi-allelic variants ─────────────────────────────────────
def expand_multiallelic(df):
    rows = []
    for _, row in df.iterrows():
        alts = str(row['ALT']).split(',')
        if len(alts) == 1:
            rows.append(row.to_dict())
        else:
            for alt in alts:
                alt = alt.strip()
                new_row = row.to_dict()
                new_row['ALT'] = alt
                new_row['type'] = variant_type(str(row['REF']), alt)
                rows.append(new_row)
    return pd.DataFrame(rows)

df_reg = expand_multiallelic(df_reg)

# Recompute variant IDs after expansion
df_reg['variant_id'] = (df_reg['CHROM'].astype(str) + '_' +
                        df_reg['POS'].astype(str)   + '_' +
                        df_reg['REF'].astype(str)   + '_' +
                        df_reg['ALT'].astype(str))

df_reg = df_reg.drop_duplicates(subset='variant_id').reset_index(drop=True)
print(f"After multi-allelic expansion: {len(df_reg)} variants")
print(f"  High-Background: {(df_reg['group']=='High-Background').sum()}")
print(f"  Low-Background:  {(df_reg['group']=='Low-Background').sum()}")

df_reg['variant_id'] = (df_reg['CHROM'].astype(str) + '_' +
                        df_reg['POS'].astype(str)   + '_' +
                        df_reg['REF'].astype(str)   + '_' +
                        df_reg['ALT'].astype(str))

# ── Filter invalid bases ──────────────────────────────────────────────
valid_bases = set('ACGTN')
def is_valid(ref, alt):
    return (set(str(ref)).issubset(valid_bases) and
            set(str(alt)).issubset(valid_bases))

before = len(df_reg)
df_reg = df_reg[df_reg.apply(lambda r: is_valid(r['REF'], r['ALT']), axis=1)].reset_index(drop=True)
print(f"After base validation filter: {len(df_reg)} variants (removed {before - len(df_reg)})")

# Drop duplicates just in case
df_reg = df_reg.drop_duplicates(subset='variant_id').reset_index(drop=True)
print(f"After deduplication: {len(df_reg)} variants")
print(f"  High-Background: {(df_reg['group']=='High-Background').sum()}")
print(f"  Low-Background:  {(df_reg['group']=='Low-Background').sum()}")
print(f"  SNVs: {(df_reg['type']=='SNV').sum()}, Insertions: {(df_reg['type']=='Insertion').sum()}, Deletions: {(df_reg['type']=='Deletion').sum()}")

# ── Load checkpoint ───────────────────────────────────────────────────
def load_checkpoint():
    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, 'r') as f:
            return json.load(f)
    return {"processed_ids": [], "results": []}

def save_checkpoint(processed_ids, results):
    with open(CHECKPOINT_FILE, 'w') as f:
        json.dump({"processed_ids": processed_ids, "results": results}, f)
    # Also save running CSV to Drive
    if results:
        pd.DataFrame(results).to_csv(RESULTS_FILE, index=False)

checkpoint    = load_checkpoint()
processed_ids = set(checkpoint["processed_ids"])
results       = checkpoint["results"]

skipped = len(processed_ids)
if skipped > 0:
    print(f"\n[RESUME] Found checkpoint: {skipped} variants already processed, resuming...")
else:
    print("\n[FRESH] No checkpoint found, starting from scratch...")

# ── Helper: run one variant through AlphaGenome ──────────────────────
def run_variant(chrom, pos, ref, alt):
    start_pos   = max(0, pos - (ONE_MB // 2))
    end_pos     = start_pos + ONE_MB
    interval    = genome.Interval(chromosome=chrom, start=start_pos, end=end_pos)
    variant_obj = genome.Variant(
        chromosome=chrom, position=pos,
        reference_bases=str(ref), alternate_bases=str(alt),
    )

    scores = {}
    for cfg in MODALITY_CONFIG:
        label    = cfg["label"]
        modality = cfg["modality"]
        attr     = cfg["attr"]
        terms    = cfg["ontology_terms"]

        try:
            preds    = model.predict_variant(
                interval=interval,
                variant=variant_obj,
                organism=dna_client.Organism.HOMO_SAPIENS,
                requested_outputs=[modality],
                ontology_terms=terms,
            )
            ref_vals = getattr(preds.reference, attr).values
            alt_vals = getattr(preds.alternate, attr).values

            if ref_vals.shape[1] == 0 or alt_vals.shape[1] == 0:
                scores[f"αG_{label}_LFC_mean"] = np.nan
                scores[f"αG_{label}_LFC_max"]  = np.nan
                continue

            eps                  = 1e-5
            ref_peak             = ref_vals.max(axis=0)
            alt_peak             = alt_vals.max(axis=0)
            valid                = ref_peak > 0.01
            if valid.sum() == 0:
                scores[f"αG_{label}_LFC_mean"] = np.nan
                scores[f"αG_{label}_LFC_max"]  = np.nan
                continue

            lfc                            = np.log2((alt_peak[valid] + eps) /
                                                     (ref_peak[valid] + eps))
            scores[f"αG_{label}_LFC_mean"] = round(float(np.mean(lfc)), 4)
            scores[f"αG_{label}_LFC_max"]  = round(float(np.max(np.abs(lfc))), 4)

        except Exception as e:
            scores[f"αG_{label}_LFC_mean"] = np.nan
            scores[f"αG_{label}_LFC_max"]  = np.nan

    return scores

# ── Main loop ─────────────────────────────────────────────────────────
variants_to_run = df_reg[~df_reg['variant_id'].isin(processed_ids)]
total           = len(variants_to_run)
print(f"\nVariants to process: {total}")

for i, (_, row) in enumerate(variants_to_run.iterrows()):
    v_id  = row['variant_id']
    chrom = str(row['CHROM'])
    pos   = int(row['POS'])
    ref   = str(row['REF'])
    alt   = str(row['ALT'])
    gene  = str(row['SYMBOL'])
    group = row['group']
    vtype = row['type']

    print(f"  [{i+1}/{total}] {gene} {chrom}:{pos} ({vtype}, {group})...", end=' ')

    scores = run_variant(chrom, pos, ref, alt)

    record = {
        "variant_id":    v_id,
        "gene":          gene,
        "group":         group,
        "coordinates":   f"{chrom}:{pos}",
        "type":          vtype,
        "log2FC":        row.get('log2FoldChange', np.nan),
        "padj":          row.get('padj', np.nan),
        "cCRE_Type":     row.get('cCRE_Type', ''),
        **scores,
    }

    # Compute total impact score
    max_cols = [k for k in scores if 'LFC_max' in k]
    record["αG_Total_Impact_Score"] = round(
        float(np.nansum([scores[k] for k in max_cols])), 4
    )

    results.append(record)
    processed_ids.add(v_id)

    total_score = record["αG_Total_Impact_Score"]
    print(f"score={total_score:.4f}")

    # Checkpoint every N variants
    if (i + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(list(processed_ids), results)
        print(f"    [CHECKPOINT SAVED] {len(processed_ids)} variants complete")

# Final save
save_checkpoint(list(processed_ids), results)
print(f"\n[DONE] All {len(processed_ids)} background variants processed.")
